In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('hygdata_v37.csv.gz', compression='gzip')

print(f"Stars loaded: {len(df):,}")
print(f"\nColumns:\n{list(df.columns)}")

Stars loaded: 119,626

Columns:
['id', 'hip', 'hd', 'hr', 'gl', 'bf', 'proper', 'ra', 'dec', 'dist', 'pmra', 'pmdec', 'rv', 'mag', 'absmag', 'spect', 'ci', 'x', 'y', 'z', 'vx', 'vy', 'vz', 'rarad', 'decrad', 'pmrarad', 'pmdecrad', 'bayer', 'flam', 'con', 'comp', 'comp_primary', 'base', 'lum', 'var', 'var_min', 'var_max']


In [4]:
import urllib.request

url = 'https://www.astronexus.com/downloads/catalogs/hygdata_v37.csv.gz'
urllib.request.urlretrieve(url, 'hygdata_v37.csv.gz')
print('Downloaded.')


Downloaded.


In [6]:
# Keep only stars with valid colour index and absolute magnitude
hr = df[['proper', 'absmag', 'ci', 'spect', 'lum', 'dist', 'mag']].copy()

# Drop rows missing the two key columns
hr = hr.dropna(subset=['absmag', 'ci'])

# Drop stars with bad distance data (dist >= 100000 means missing parallax)
hr = hr[df['dist'] < 100000]

# Reset index
hr = hr.reset_index(drop=True)

print(f"Stars after cleaning: {len(hr):,}")
print(f"\nSample:\n{hr.head()}")
print(f"\nci range: {hr.ci.min():.2f} to {hr.ci.max():.2f}")
print(f"absmag range: {hr.absmag.min():.2f} to {hr.absmag.max():.2f}")

Stars after cleaning: 107,860

Sample:
  proper  absmag     ci spect         lum      dist    mag
0    Sol   4.850  0.656   G2V    1.000000    0.0000 -26.70
1    NaN   2.390  0.482    F5    9.638290  219.7802   9.10
2    NaN   5.866  0.999   K3V    0.392283   47.9616   9.27
3    NaN  -1.619 -0.019    B9  386.901132  442.4779   6.61
4    NaN   2.421  0.370   F0V    9.366989  134.2282   8.06

ci range: -0.40 to 5.46
absmag range: -7.22 to 18.68


C:\Users\james\AppData\Local\Temp\ipykernel_26992\522960232.py:8: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  hr = hr[df['dist'] < 100000]


In [8]:
def ci_to_hex(ci):
    if ci < -0.30: return '#9bb0ff'
    elif ci < 0.00: return '#aabfff'
    elif ci < 0.15: return '#cad7ff'
    elif ci < 0.30: return '#f8f7ff'
    elif ci < 0.58: return '#fff4ea'
    elif ci < 0.81: return '#fffcea'
    elif ci < 1.40: return '#ffd2a1'
    else:           return '#ffad51'

hr['color'] = hr['ci'].apply(ci_to_hex)

def spect_label(s):
    if pd.isna(s) or len(str(s)) == 0:
        return 'Unknown'
    return str(s)[0]

hr['spect_class'] = hr['spect'].apply(spect_label)
hr['label'] = hr['proper'].fillna('')

print("Colour mapping done.")
print(hr['spect_class'].value_counts().head(10))

Colour mapping done.
spect_class
K          28845
F          24594
G          21539
A          17482
B           8206
M           4248
Unknown     2006
m            228
D            154
O            115
Name: count, dtype: int64


In [9]:
import plotly.graph_objects as go

# Downsample slightly for performance - 50k points is plenty
sample = hr.sample(n=50000, random_state=42)

# Highlight named stars separately
named = hr[hr['label'] != '']

fig = go.Figure()

# Main star field
fig.add_trace(go.Scatter(
    x=sample['ci'],
    y=sample['absmag'],
    mode='markers',
    marker=dict(
        color=sample['color'],
        size=2,
        opacity=0.6,
    ),
    hovertemplate=(
        'Colour Index: %{x:.3f}<br>'
        'Abs. Magnitude: %{y:.2f}<br>'
        'Spectral: %{customdata}'
        '<extra></extra>'
    ),
    customdata=sample['spect'],
    name='Stars'
))

# Named stars on top
fig.add_trace(go.Scatter(
    x=named['ci'],
    y=named['absmag'],
    mode='markers+text',
    marker=dict(
        color=named['color'],
        size=6,
        opacity=1.0,
        line=dict(color='white', width=0.5)
    ),
    text=named['label'],
    textposition='top right',
    textfont=dict(color='white', size=9),
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Colour Index: %{x:.3f}<br>'
        'Abs. Magnitude: %{y:.2f}<br>'
        'Spectral: %{customdata}'
        '<extra></extra>'
    ),
    customdata=named['spect'],
    name='Named Stars'
))

fig.update_layout(
    title=dict(
        text='Hertzsprung-Russell Diagram  |  107,860 Stars',
        font=dict(size=20, color='white'),
        x=0.5
    ),
    paper_bgcolor='#04040f',
    plot_bgcolor='#04040f',
    xaxis=dict(
        title='Colour Index (B-V)  ←  Hot / Blue          Cool / Red  →',
        color='#888',
        gridcolor='#111128',
        zerolinecolor='#222',
        range=[-0.5, 2.5]
    ),
    yaxis=dict(
        title='Absolute Magnitude (dimmer →)',
        color='#888',
        gridcolor='#111128',
        zerolinecolor='#222',
        autorange='reversed'
    ),
    legend=dict(
        font=dict(color='white'),
        bgcolor='rgba(0,0,0,0.5)'
    ),
    height=750,
    margin=dict(l=60, r=20, t=80, b=60)
)

fig.show()

In [10]:
fig2 = go.Figure()

# Main star field
fig2.add_trace(go.Scatter(
    x=sample['ci'],
    y=sample['absmag'],
    mode='markers',
    marker=dict(color=sample['color'], size=2, opacity=0.6),
    hovertemplate=(
        'Colour Index: %{x:.3f}<br>'
        'Abs. Magnitude: %{y:.2f}<br>'
        'Spectral: %{customdata}<extra></extra>'
    ),
    customdata=sample['spect'],
    name='Stars'
))

# Named stars
fig2.add_trace(go.Scatter(
    x=named['ci'],
    y=named['absmag'],
    mode='markers+text',
    marker=dict(
        color=named['color'], size=6, opacity=1.0,
        line=dict(color='white', width=0.5)
    ),
    text=named['label'],
    textposition='top right',
    textfont=dict(color='white', size=9),
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Colour Index: %{x:.3f}<br>'
        'Abs. Magnitude: %{y:.2f}<br>'
        'Spectral: %{customdata}<extra></extra>'
    ),
    customdata=named['spect'],
    name='Named Stars'
))

# Annotation regions
regions = [
    dict(
        x=0.05, y=-6.5, text='SUPERGIANTS',
        font_size=11, font_color='#aabfff',
        showarrow=False, xref='x', yref='y',
        font=dict(family='monospace', size=11, color='#aabfff')
    ),
    dict(
        x=1.6, y=-2.5, text='RED GIANTS',
        font_size=11, font_color='#ffd2a1',
        showarrow=False, xref='x', yref='y',
        font=dict(family='monospace', size=11, color='#ffd2a1')
    ),
    dict(
        x=0.55, y=3.2, text='MAIN SEQUENCE',
        font_size=11, showarrow=False,
        xref='x', yref='y', textangle=-52,
        font=dict(family='monospace', size=11, color='rgba(255,255,255,0.4)')
    ),
    dict(
        x=-0.15, y=13.5, text='WHITE DWARFS',
        font_size=11, showarrow=False,
        xref='x', yref='y',
        font=dict(family='monospace', size=11, color='#cad7ff')
    ),
    dict(
        x=1.9, y=12.5, text='RED DWARFS',
        font_size=11, showarrow=False,
        xref='x', yref='y',
        font=dict(family='monospace', size=11, color='#ffad51')
    ),
]

# Spectral class labels along the top
spect_labels = [
    dict(x=-0.25, y=-8.2, text='O', font=dict(size=13, color='#9bb0ff', family='monospace')),
    dict(x=0.05,  y=-8.2, text='B', font=dict(size=13, color='#aabfff', family='monospace')),
    dict(x=0.25,  y=-8.2, text='A', font=dict(size=13, color='#cad7ff', family='monospace')),
    dict(x=0.45,  y=-8.2, text='F', font=dict(size=13, color='#fff4ea', family='monospace')),
    dict(x=0.70,  y=-8.2, text='G', font=dict(size=13, color='#fffcea', family='monospace')),
    dict(x=1.10,  y=-8.2, text='K', font=dict(size=13, color='#ffd2a1', family='monospace')),
    dict(x=1.60,  y=-8.2, text='M', font=dict(size=13, color='#ffad51', family='monospace')),
]

all_annotations = regions + [
    dict(showarrow=False, xref='x', yref='y', **s) for s in spect_labels
]

fig2.update_layout(
    title=dict(
        text='Hertzsprung-Russell Diagram  |  107,860 Stars',
        font=dict(size=20, color='white'),
        x=0.5
    ),
    paper_bgcolor='#04040f',
    plot_bgcolor='#04040f',
    xaxis=dict(
        title='Colour Index (B-V)     ← Hot / Blue                    Cool / Red →',
        color='#888',
        gridcolor='#111128',
        zerolinecolor='#222',
        range=[-0.5, 2.5]
    ),
    yaxis=dict(
        title='Absolute Magnitude (dimmer →)',
        color='#888',
        gridcolor='#111128',
        zerolinecolor='#222',
        autorange='reversed',
        range=[-8.5, 18]
    ),
    annotations=all_annotations,
    legend=dict(font=dict(color='white'), bgcolor='rgba(0,0,0,0.5)'),
    height=800,
    margin=dict(l=60, r=20, t=80, b=60)
)

fig2.show()

In [11]:
fig2.write_html('hr_diagram.html')
print('Exported.')

Exported.
